# Chạy LLM extractor (Qwen2.5, few-shot) trên Google Colab
Self-host ≤9B, KHÔNG API ngoài. Runtime → Change runtime type → **GPU** (T4/L4/A100).
Model lấy từ `configs/config.yaml → extract.llm_model` (mặc định **3B**; đổi **7B** nếu muốn few-shot mạnh hơn — cell 5 có sẵn dòng đổi).
Mẹo: cache model vào Google Drive (cell 1) để khỏi tải lại 15GB mỗi phiên.

In [ ]:
# 1) (Khuyến nghị) Mount Drive + cache HuggingFace vào Drive để khỏi tải lại model
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
    print('HF_HOME =', os.environ['HF_HOME'])
except Exception as e:
    print('Bỏ qua Drive:', e)

In [ ]:
# 2) Lấy code
%cd /content
![ -d viettel-medreason ] || git clone https://github.com/tienndat1904/viettel-medreason.git
%cd /content/viettel-medreason
!git pull -q

In [ ]:
# 3) Cài env — KHÔNG động vào numpy/transformers/torch (dùng bản Colab ship sẵn, đã tương thích).
#    Chỉ update bitsandbytes/peft/accelerate. (requirements.txt numpy>=1.26 nên KHÔNG hạ cấp numpy.)
!pip install -q -r requirements.txt
!pip install -q -U bitsandbytes peft accelerate
import numpy, torch, transformers, bitsandbytes as bnb
print('numpy', numpy.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| bnb', bnb.__version__, '| CUDA', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not numpy.__version__.startswith('2'):
    raise SystemExit('numpy != 2.x (session cu ban) -> Runtime > Disconnect and delete runtime, roi Run all lai')

In [ ]:
# 4) SMOKE TEST 3 file trước (kiểm model + JSON parse; lần đầu tải model ~15GB)
import os, shutil
os.makedirs('smoke/input', exist_ok=True)
for n in [1, 5, 50]:
    shutil.copy(f'data/test/input/{n}.txt', f'smoke/input/{n}.txt')
!python src/pipeline.py --input smoke/input --output smoke/out --backend llm
print('----- smoke/out/5.json -----')
!cat smoke/out/5.json

In [ ]:
# 5) Chạy full 100 file -> GHI VÀO GOOGLE DRIVE + --resume.
#    Colab ngắt/máy sleep -> chạy lại đúng cell này (sau cell 1-3): tự BỎ QUA file đã xong (đừng xoá output!).
#
# (Tùy chọn) đổi model few-shot 3B -> 7B (mạnh hơn, chậm hơn; đều ≤9B, KHÔNG train). Bỏ comment 2 dòng:
# import yaml; _c=yaml.safe_load(open('configs/config.yaml',encoding='utf-8')); _c['extract']['llm_model']='Qwen/Qwen2.5-7B-Instruct'
# yaml.safe_dump(_c, open('configs/config.yaml','w',encoding='utf-8'), allow_unicode=True, sort_keys=False)
OUT = '/content/drive/MyDrive/vmr_output'
# ⚠️ CHỈ bỏ comment khi ĐỔI MODEL / muốn chạy lại SẠCH (KHÔNG bỏ comment khi resume sau khi Colab ngắt):
# !rm -rf {OUT}
!python src/pipeline.py --input data/test/input --output {OUT} --backend llm --resume
!ls {OUT} | wc -l   # số file đã xong / 100

In [ ]:
# 6) Validate + đóng gói (đọc output từ Drive) + tải về
OUT = '/content/drive/MyDrive/vmr_output'
!python scripts/package_submission.py --output {OUT} --input data/test/input --n 100
from google.colab import files; files.download('output.zip')

In [ ]:
# 7) Đo trên dev gold (copy output 2 cell này gửi lại để tinh chỉnh)
OUT = '/content/drive/MyDrive/vmr_output'
!python src/eval/official_scorer.py --pred {OUT} --gold data/dev/gold   # METRIC CHÍNH THỨC BTC
!python src/eval/scorer.py --pred {OUT} --gold data/dev/gold --mode overlap   # tham khảo (span-F1)
!python src/eval/eval_linking.py